In [1]:
import torch
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
device

'cuda'

In [2]:
x = torch.tensor(1.2, requires_grad=True)
y = torch.tensor(3.4, requires_grad=True)
f = torch.sin(x**2*y)
f

tensor(-0.9832, grad_fn=<SinBackward0>)

In [3]:
f.backward()

In [4]:
print(x.grad,y.grad)

tensor(1.4899) tensor(0.2629)


In [5]:
def f(x,y):
    return torch.sin(x ** 2 * y)

x = torch.tensor(1.2, requires_grad=True)
y - torch.tensor(3.4, requires_grad=True)
result = f(x,y)
result.backward()

x.grad.item(),y.grad.item()

(1.489864706993103, 0.5258346199989319)

In [6]:
def f_vec(v):
    return torch.sin(v[0] ** 2 * v[1])
v = torch.tensor([1.2,3.4], requires_grad=True)
result = f_vec(v)
result.backward()

v.grad

tensor([1.4899, 0.2629])

In [9]:
import torch.nn as nn
class Dense(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.relu = nn.ReLU()
    
    def forward(self, X):
        return self.relu(self.linear(X))

In [10]:
torch.manual_seed(42)
dense = Dense(3,5)
X = torch.randn(2,3)
y_pred = dense(X)
y_pred.shape

torch.Size([2, 5])

In [11]:
y_pred_check = dense.relu(X @ dense.linear.weight.T + dense.linear.bias)
torch.allclose(y_pred, y_pred_check)

True

In [13]:
import torch.nn.functional as F
class Dense2(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
    
    def forward(self, X):
        z = X @ self.weight.T + self.bias
        return F.relu(z)

In [14]:
torch.manual_seed(42)
dense2 = Dense2(3, 5)
X = torch.randn(2,3)
y_pred2 = dense2(X)
y_pred2.shape

torch.Size([2, 5])

In [15]:
y_pred2_check = F.relu(X @ dense2.weight.T + dense2.bias)
torch.allclose(y_pred2, y_pred2_check)

True

In [16]:
from sklearn.datasets import fetch_covtype
from torch.utils.data import TensorDataset

covtype = fetch_covtype()

X_covtype = torch.tensor(covtype.data, dtype=torch.float32)
means = X_covtype.mean(dim =0 , keepdim=True)
stds = X_covtype.std(dim=0, keepdim=True)
X_standardized_covtype = (X_covtype - means) / stds

y_cotype = torch.tensor(covtype.target - 1, dtype=torch.long)

covtype_dataset = TensorDataset(X_standardized_covtype, y_cotype)

In [17]:
from torch.utils.data import random_split

torch.manual_seed(42)

train_size = len(covtype_dataset) * 80 // 100
valid_size = len(covtype_dataset) * 10 // 100
test_size = len(covtype_dataset) - train_size - valid_size

train_dataset, valid_dataset, test_dataset = random_split(
    covtype_dataset,
    [train_size, valid_size, test_size]
)

In [18]:
from torch.utils.data import DataLoader

batch_size = 256

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [23]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [24]:
def train2(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [19]:
n_inputs = len(covtype.feature_names)
n_classes = len(set(covtype.target))

torch.manual_seed(42)
covtype_model = nn.Sequential(
    nn.Linear(n_inputs, 200), nn.ReLU(),
    nn.Linear(200, 100), nn.ReLU(),
    nn.Linear(100, 50), nn.ReLU(),
    nn.Linear(50, n_classes)
).to(device)

In [26]:
torch.manual_seed(42)

for learning_rate in [0.16, 0.08, 0.04, 0.02, 0.01, 0.005]:
    n_epochs = 15
    optimizer = torch.optim.SGD(covtype_model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    metric = torchmetrics.Accuracy(task='multiclass',
                                   num_classes= n_classes).to(device)
    history= train2(covtype_model, optimizer, criterion, metric, train_loader, valid_loader,
                    n_epochs)

Epoch 1/15, train loss: 0.6336, train metric: 0.7343, valid metric: 0.7627
Epoch 2/15, train loss: 0.5142, train metric: 0.7787, valid metric: 0.8022
Epoch 3/15, train loss: 0.4555, train metric: 0.8053, valid metric: 0.8187
Epoch 4/15, train loss: 0.4116, train metric: 0.8264, valid metric: 0.8312
Epoch 5/15, train loss: 0.3807, train metric: 0.8404, valid metric: 0.8030
Epoch 6/15, train loss: 0.3552, train metric: 0.8520, valid metric: 0.8581
Epoch 7/15, train loss: 0.3324, train metric: 0.8625, valid metric: 0.8653
Epoch 8/15, train loss: 0.3170, train metric: 0.8692, valid metric: 0.8770
Epoch 9/15, train loss: 0.3011, train metric: 0.8764, valid metric: 0.8647
Epoch 10/15, train loss: 0.2874, train metric: 0.8824, valid metric: 0.8787
Epoch 11/15, train loss: 0.2762, train metric: 0.8869, valid metric: 0.8904
Epoch 12/15, train loss: 0.2661, train metric: 0.8915, valid metric: 0.8893
Epoch 13/15, train loss: 0.2574, train metric: 0.8945, valid metric: 0.8909
Epoch 14/15, train lo

In [27]:
class Dense3(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, nonlinearity="relu")
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, X):
        z = X @ self.weight.T + self.bias
        return F.relu(z)

In [28]:
class CoverTypeModel(nn.Module):
    def __init__(self, n_neurons, n_inputs=n_inputs, n_classes=n_classes):
        super().__init__()
        layers = [
            Dense3(n_in, n_out)
            for n_in, n_out in zip([n_inputs] + n_neurons, n_neurons)
        ] + [nn.Linear(n_neurons[-1], n_classes)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, X):
        return self.mlp(X)


In [30]:
%pip install optuna -q
import optuna
def objective(trial):
    learning_rate = trial.suggest_float('learning_rate',1e-2, 1.0, log=True)
    n_layers = trial.suggest_int('n_layers',1,3)
    n_hidden = trial.suggest_int('n_hidden', 30, 150)
    covtype_model = CoverTypeModel([n_hidden] * n_layers).to(device)
    optimizer = torch.optim.SGD(covtype_model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task='multiclass',
                                     num_classes=n_classes).to(device)
    best_validation_accuracy = 0.0
    for epoch in range(n_epochs):
        history = train2(covtype_model, optimizer, xentropy, accuracy,
                         train_loader, valid_loader, n_epochs=1)
        validation_accuracy= max(history['valid_metrics'])
        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
        trial.report(validation_accuracy, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_validation_accuracy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.8 MB/s eta 0:00:00


In [33]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner()
study = optuna.create_study(direction='maximize', sampler=sampler,
                            pruner=pruner)
study.optimize(objective, n_trials=2)

[I 2026-03-17 07:00:44,089] A new study created in memory with name: no-name-f9838ad3-e9f5-4e09-9f75-0d34e632453a


Epoch 1/1, train loss: 0.6380, train metric: 0.7309, valid metric: 0.7675
Epoch 1/1, train loss: 0.5247, train metric: 0.7761, valid metric: 0.7925
Epoch 1/1, train loss: 0.4781, train metric: 0.7977, valid metric: 0.8016
Epoch 1/1, train loss: 0.4443, train metric: 0.8131, valid metric: 0.8176
Epoch 1/1, train loss: 0.4184, train metric: 0.8249, valid metric: 0.8187
Epoch 1/1, train loss: 0.3973, train metric: 0.8344, valid metric: 0.8435
Epoch 1/1, train loss: 0.3794, train metric: 0.8424, valid metric: 0.8473
Epoch 1/1, train loss: 0.3626, train metric: 0.8499, valid metric: 0.8365
Epoch 1/1, train loss: 0.3483, train metric: 0.8570, valid metric: 0.8578
Epoch 1/1, train loss: 0.3374, train metric: 0.8615, valid metric: 0.8682
Epoch 1/1, train loss: 0.3256, train metric: 0.8664, valid metric: 0.8697
Epoch 1/1, train loss: 0.3164, train metric: 0.8707, valid metric: 0.8718
Epoch 1/1, train loss: 0.3067, train metric: 0.8753, valid metric: 0.8705
Epoch 1/1, train loss: 0.3009, train m

[I 2026-03-17 07:02:43,470] Trial 0 finished with value: 0.8718094229698181 and parameters: {'learning_rate': 0.05611516415334506, 'n_layers': 3, 'n_hidden': 118}. Best is trial 0 with value: 0.8718094229698181.


Epoch 1/1, train loss: 0.2921, train metric: 0.8812, valid metric: 0.8715
Epoch 1/1, train loss: 0.6367, train metric: 0.7331, valid metric: 0.7502
Epoch 1/1, train loss: 0.5669, train metric: 0.7583, valid metric: 0.7629
Epoch 1/1, train loss: 0.5448, train metric: 0.7674, valid metric: 0.7767
Epoch 1/1, train loss: 0.5295, train metric: 0.7747, valid metric: 0.7790
Epoch 1/1, train loss: 0.5189, train metric: 0.7787, valid metric: 0.7722
Epoch 1/1, train loss: 0.5111, train metric: 0.7828, valid metric: 0.7893
Epoch 1/1, train loss: 0.5050, train metric: 0.7857, valid metric: 0.7893
Epoch 1/1, train loss: 0.4998, train metric: 0.7877, valid metric: 0.7916
Epoch 1/1, train loss: 0.4956, train metric: 0.7889, valid metric: 0.7943
Epoch 1/1, train loss: 0.4918, train metric: 0.7910, valid metric: 0.7893
Epoch 1/1, train loss: 0.4887, train metric: 0.7927, valid metric: 0.8006
Epoch 1/1, train loss: 0.4849, train metric: 0.7940, valid metric: 0.7938
Epoch 1/1, train loss: 0.4839, train m

[I 2026-03-17 07:04:27,956] Trial 1 finished with value: 0.8006230592727661 and parameters: {'learning_rate': 0.15751320499779725, 'n_layers': 1, 'n_hidden': 48}. Best is trial 0 with value: 0.8718094229698181.


Epoch 1/1, train loss: 0.4759, train metric: 0.7982, valid metric: 0.7913


In [34]:
study.best_value

0.8718094229698181

In [35]:
study.best_params

{'learning_rate': 0.05611516415334506, 'n_layers': 3, 'n_hidden': 118}